In [ ]:
import pandas as pd

# Path to data frame containing all WSIs with a matched rekvnr
df_path = r"D:\DATA\full_dataset\overlapping_rekvnr.csv"
df_overlapping = pd.read_csv(df_path)

In [ ]:
print(df_overlapping.columns)
print(df_overlapping.shape)

In [ ]:
# Remove columns with no additional information
cols_to_drop = ["serviceyder", "undafd", "mattype"]
# serviceyder (3800Q80), undafd (Næstved3 Patologiafdeling), all samples prepared at same place
# mattype (redundant, same as "mattype tekst")

df_overlapping = df_overlapping.drop(columns=cols_to_drop)
print(df_overlapping.shape)

In [ ]:
print(len(df_overlapping["rekvnr"].unique()))

In [ ]:
# Drop rows where rekvnr is missing
before = len(df_overlapping)
df_overlapping = df_overlapping.dropna(subset=["rekvnr"])
after = len(df_overlapping)

dropped = before - after
print("Number of rows dropped (no pathology rekvnr):", dropped)

In [ ]:
print("Number of duplicated rows:", df_overlapping.duplicated().sum())

In [ ]:
# Duplicate values in filename

# Mask for rows with duplicate rekvnr
dup_mask = df_overlapping["filename"].duplicated(keep=False)
df_dups = df_overlapping[dup_mask]

num_dups = df_overlapping["filename"].duplicated().sum()
print("Number of duplicate filenames:", num_dups)

In [ ]:
from helper_functions import combine_values

# Combine rows with same filename
df_no_dups = df_overlapping.groupby(["filename"], as_index=False).agg(combine_values)

# Get results
print("Before combing duplicated filename:", len(df_overlapping))
print("After combining rows with duplicate filename:", len(df_no_dups))

In [ ]:
# Path to SNOMED codes (all codes with code history)
snomed_path = "D:\DATA\patoSnoMed_2025-04.xlsx"
df_snomed = pd.read_excel(snomed_path)

In [ ]:
from snomed_hierarchy import SNOMEDCodes

snomed = SNOMEDCodes(df_snomed)
snomed_set = snomed.snomed_set()
snomed_dict = snomed.code_to_text()

In [ ]:
from helper_functions import extract_snomed_from_all_columns

text_columns = ["snomed kode", "makrotekst", "mikrotekst", "kontekst", "kode fritekst", "glasalm", "immunalm", "ishutaelalm"]

df_no_dups["snomed_code"] = df_no_dups.apply(
    lambda r: extract_snomed_from_all_columns(r, text_columns, snomed_set),
    axis=1
)

In [ ]:
df_no_dups["M"] = df_no_dups["snomed_code"].apply(
    lambda lst: [x for x in lst if x.startswith("M")]
)
df_no_dups["T"] = df_no_dups["snomed_code"].apply(
    lambda lst: [x for x in lst if x.startswith("T")]
)
df_no_dups["F"] = df_no_dups["snomed_code"].apply(
    lambda lst: [x for x in lst if x.startswith("F")]
)
df_no_dups["P"] = df_no_dups["snomed_code"].apply(
    lambda lst: [x for x in lst if x.startswith("P")]
)
df_no_dups["S"] = df_no_dups["snomed_code"].apply(
    lambda lst: [x for x in lst if x.startswith("S")]
)
df_no_dups["Æ"] = df_no_dups["snomed_code"].apply(
    lambda lst: [x for x in lst if x.startswith("Æ")]
)

In [ ]:
from helper_functions import is_missing

# Check missing T and M codes
missing_mask_T = df_no_dups["T"].apply(is_missing)
missing_mask_M = df_no_dups["M"].apply(is_missing)

print("Number of missing values in T column:", missing_mask_T.sum())
print("Number of missing values in M column:", missing_mask_M.sum())

In [ ]:
# Find and remove rows with missing T or M
missing_snomed = df_no_dups[df_no_dups['T'].apply(is_missing) | df_no_dups['M'].apply(is_missing)]
df_no_dups = df_no_dups.drop(missing_snomed.index)

print("Number of rows dropped (no valid T / M kode):", len(missing_snomed))

In [ ]:
def snomed_codes_to_texts(codes, lookup):
    if not codes:
        return []
    return [lookup[c] for c in codes if c in lookup]

df_no_dups["snomed_text"] = df_no_dups["snomed_code"].apply(
    lambda codes: snomed_codes_to_texts(codes, snomed_dict)
)
df_no_dups["T_text"] = df_no_dups["T"].apply(
    lambda codes: snomed_codes_to_texts(codes, snomed_dict)
)
df_no_dups["M_text"] = df_no_dups["M"].apply(
    lambda codes: snomed_codes_to_texts(codes, snomed_dict)
)
df_no_dups["F_text"] = df_no_dups["F"].apply(
    lambda codes: snomed_codes_to_texts(codes, snomed_dict)
)
df_no_dups["P_text"] = df_no_dups["P"].apply(
    lambda codes: snomed_codes_to_texts(codes, snomed_dict)
)
df_no_dups["S_text"] = df_no_dups["S"].apply(
    lambda codes: snomed_codes_to_texts(codes, snomed_dict)
)
df_no_dups["Æ_text"] = df_no_dups["Æ"].apply(
    lambda codes: snomed_codes_to_texts(codes, snomed_dict)
)

In [ ]:
print(df_no_dups.head())

In [ ]:
from helper_functions import find_undersoeger

df_no_dups["undersoeger"] = df_no_dups.apply(lambda r: find_undersoeger(r, text_columns), axis=1)

In [ ]:
name_to_letter = {
    # Dict mapping names to letters for anonymization
}

def anonymize_names(name_list, mapping):
    return [mapping.get(n, n) for n in name_list]

df_no_dups["undersoeger_anonymous"] = df_no_dups["undersoeger"].apply(
    lambda x: anonymize_names(x, name_to_letter)
)

In [ ]:
cols_to_drop = ["snomed kode", "undersoeger"]
df_clean = df_no_dups.drop(columns=cols_to_drop)

In [ ]:
print(df_clean.head())

In [ ]:
from helper_functions import lists2tuples

# Lists to tuples
df_clean = lists2tuples(df_clean)

str_cols = ["filename", "rekvprio", "team", "sex", "alder gruppe", "mattype tekst", "stain"]
date_cols = ["modtdato", "rekvdato"]
int_cols = ["rekvnr", "alder", "matantal", "glasalm", "immunalm", "ishutaelalm", "antglas", "specfarv"]

# Numeric to int
for col in int_cols: 
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").astype("Int64") 

In [ ]:
import re 

# Check correct formatting
def str_pattern(col):
    if col == "rekvnr": 
        return r"\d{8}" # 8 digits
    elif col == "modtdato" or col == "rekvdato":
        return r"\d{4}-\d{2}-\d{2}" # YYYY-MM-DD, str/date
    elif col == "rekvprio":
        return r"[A-Z]{2}" # 2 uppercase letters, str
    elif col == "team":
        return r"[A-ZÆØÅ]{3}" # 3 uppercase letters, str
    elif col == "sex":
        return r"[FMfm]" # F or M, str
    elif col == "undersoeger_anonymous":
        return r"[A-Z]"  # single uppercase letter
    else:
        return None

def is_correct(value, pattern):
    if pattern is None or pd.isna(value):
        return False
        
    if isinstance(value, (list, tuple)):
        return all(
            isinstance(item, str) and
            re.fullmatch(pattern, item.strip())
            for item in value
        )

    return bool(re.fullmatch(pattern, str(value).strip()))

In [ ]:
# CHECK STR COLUMNS
for col in df_clean.columns:
    pattern = str_pattern(col)
    if pattern is None:
        continue
    incorrect_mask = ~df_clean[col].apply(lambda x: is_correct(x, pattern))
    incorrect_count = incorrect_mask.sum()
    print(f"Number of {col} with incorrect formatting: {incorrect_count}")
    if incorrect_count > 0:
        print(df_clean.loc[incorrect_mask, col].tolist())
        print("-" * 40)

In [ ]:
pattern = re.compile(r"[A-Z]")

def clean_undersoeger(value):
    if isinstance(value, tuple):
        cleaned = [
            item for item in value
            if isinstance(item, str) and pattern.fullmatch(item)
        ]
        return cleaned
    return pd.NA

df_clean["undersoeger_anonymous"] = (
    df_clean["undersoeger_anonymous"]
    .apply(clean_undersoeger)
)

In [ ]:
df_clean["undersoeger_anonymous"].value_counts()

In [ ]:
# CHECK NUMERIC COLUMNS

# Age check
invalid_age = df_clean[(df_clean['alder'] < 0) | (df_clean['alder'] > 120)]
print("Rows with invalid age: (<0 or >120)", len(invalid_age))

# numeric cols should be >= 0
for col in int_cols:
    invalid_mask = df_clean[df_clean[col]<0]
    print(f"Column: {col}, negative count:", len(invalid_mask))

In [ ]:
# CHECK DATES
for col in date_cols:
    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')

# Only allows dates from 2011 - 2013
mask_valid_dates = (df_clean['modtdato'].dt.year >= 2011) & (df_clean['modtdato'].dt.year <= 2013)
invalid_dates = df_clean[~mask_valid_dates]
print("Rows with modtdato outside 2011-2013:", len(invalid_dates))

# MODTDATO must be after REKVDATO
mask_modtdato_before_rekvdato = df_clean['modtdato'] < df_clean['rekvdato']
invalid_modtdato = df_clean[mask_modtdato_before_rekvdato]
print("Rows with modtdato later than rekvdato:", len(invalid_modtdato))

In [ ]:
from helper_functions import tuples2lists

df_clean = tuples2lists(df_clean)
print("\nData frame shape: ", df_clean.shape)
print(df_clean.head(2))

In [ ]:
# Save to csv
output_file = r"D:\DATA\full_dataset\df_cleaned.csv"
df_clean.to_csv(output_file, index=False)

print(f"Saved DataFrame to {output_file}")